## Final check: March 2024 (held back, scored once)
Saved release-2 model, no retraining, same cleaning and features through `src/`. Reported whatever it says.

In [1]:
%cd ..
from pathlib import Path

ROOT = Path.cwd()

from src import clean_data, evaluate, utils, validate
from src import main as pipeline

settings = pipeline.SETTINGS
target = settings["problem"]["target_column"]
model = utils.load_model(ROOT / settings["paths"]["model"])
zones = utils.load_csv(ROOT / settings["paths"]["zone_table"])
march_raw = utils.load_csv(ROOT / "data/final_check/yellow_trips_2024_03.csv")

march = clean_data.clean_dataframe(
    march_raw, target_column=target, zone_table=zones, rules=settings["cleaning"]
)
validate.validate_dataframe(march, **pipeline._validation_arguments(training=True))
metrics = evaluate.evaluate_model(
    model, march[pipeline._feature_columns()], march[target],
    settings["problem"]["problem_type"], **settings["business"],
)
status = pipeline.check_quality_gates(metrics, settings["quality_gates"])

print(f"March 2024: {len(march_raw):,} trips, {len(march):,} after cleaning")
for key in ("mae", "rmse", "r2", "bias", "on_time_share", "big_miss_share"):
    print(f"{key}: {metrics[key]:.3f}" + (f" -> {status[key].upper()}" if key in status else ""))

Dropped 767 of 20000 training rows (cleaning rules)


March 2024: 20,000 trips, 19,233 after cleaning
mae: 3.793 -> MET
rmse: 6.225
r2: 0.765
bias: -1.125 -> NOT MET
on_time_share: 0.576 -> NOT MET
big_miss_share: 0.067 -> MET
